# Notebook 02 - Features (Week 2)

Convert the event stream into one interpretable feature row per incident,
check class balance, and inspect how each feature separates the classes.

**Deliverable:** `features.csv` with no secrets/PII, and a documented feature
meaning in `docs/features.md`.

The canonical version of this logic lives in `ml/features.py` so it is
reproducible outside the notebook: `python ml/features.py`

In [ ]:
import sys
from pathlib import Path


def _package_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / 'sentinelllm' / 'analysis').is_dir():
            return base / 'sentinelllm'
        if base.name == 'sentinelllm' and (base / 'analysis').is_dir():
            return base
    raise RuntimeError('Run this notebook from inside the sentinelllm/ tree')


PACKAGE_ROOT = _package_root()
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

import pandas as pd

from analysis.load_events import load_events, load_labels
from ml.features import FEATURE_COLUMNS, extract_features

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

## Build the feature table

In [ ]:
events = load_events()
features = extract_features(events)
print('feature table:', features.shape)
features.head()

## Class balance

Malicious incidents are the minority, so accuracy is meaningless here. Week 3-4
will report precision, recall, F1, PR-AUC and false-positive rate, and may use
`class_weight='balanced'`.

In [ ]:
features['label'].value_counts().to_frame('incidents')

## Feature prevalence by label

Each column is a 0/1 (or count). This shows which features carry signal.

In [ ]:
binary = [c for c in FEATURE_COLUMNS if c not in ('incident_id', 'label')]
features.groupby('label')[binary].mean().T

## The false-positive trap

`ambiguous_unsigned_tool` incidents look risky (unsigned binary in Temp) but are
benign. A detector that fires on `has_unsigned_process` or `temp_path` alone will
quarantine legitimate tools - exactly the failure mode the proposal warns about.
Notice they lack the corroborating signals (credential access, rare domain, AV hit).

In [ ]:
manifest = load_labels()
merged = features.merge(manifest[['incident_id', 'scenario']], on='incident_id')
merged.groupby('scenario')[['has_unsigned_process', 'temp_path', 'browser_store_access', 'rare_domain', 'av_hit']].mean()

## Week 2 checklist

- [x] One row per incident
- [x] Binary/interpretable features first
- [x] Feature extraction lives in `ml/features.py` (not only in the notebook)
- [x] No secrets/PII in the feature table
- [x] Feature meaning documented in `docs/features.md`